## Gaussian Process Regression

Gaussian Process Regression (GPR) is a non-parametric, Bayesian approach to regression that is particularly useful for modeling complex, non-linear relationships in data. It provides a probabilistic framework for making predictions and quantifying uncertainty.

- It is particularly effective when the relationship between the input and output variables is complex and non-linear.
- GPR provides a measure of uncertainty for each prediction, which is useful for decision-making.
- It can handle small datasets well, as it does not rely on a large amount of data to make accurate predictions.

### Load data & Initialization

#### imports

In [106]:
# --------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Pipeline & joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib

# Plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

#### Google colab setup from vscode using colabsync

make sure to connect the colab sync

In [107]:


#loading .env file for notebooks, it looks for .env file in the current working directory
import os
os.chdir("/content")
print("Current working directory: ", os.getcwd())
#list all files in the current working directory
print("Files in current working directory: ", os.listdir(os.getcwd()))

#change the current working directory to the directory of the notebook
os.chdir("/content/international-project/FSIQ-from-ECG/Group A/Regression Method 3 Gaussian")

print("Current working directory: ", os.getcwd())
print("Files in current working directory: ", os.listdir(os.getcwd()))

%load_ext dotenv
%reload_ext dotenv
%dotenv

#reset environment variables to avoid conflicts
os.environ.pop("DATA_FILE", None)
%dotenv

Current working directory:  /content
Files in current working directory:  ['.config', 'international-project', 'sample_data']
Current working directory:  /content/international-project/FSIQ-from-ECG/Group A/Regression Method 3 Gaussian
Files in current working directory:  ['.env', 'method3.ipynb']
The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


#### loading data from .env file

In [108]:
data_file = os.getenv("DATA_FILE")
non_linearity_file = os.getenv("NON_LINEARITY_FILE")

def load_data(file_path):
    try:
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path, sep="\t", encoding="utf-8-sig", header=None) #change the delimiter to your needs.
        df = df.replace({ #fixing accents, the original file has some encoding issues, so we replace the incorrect characters with the correct ones, we cannot recover automatically
                    "Num�ro": "Numéro",
                    "Diab�te": "Diabète",
                    "Pr�matur�": "Prématuré",
                })
        
        column_names = df.iloc[0].tolist()
        descriptions = df.iloc[1].fillna("").tolist()
        descriptions = ["","",""] + descriptions
        descriptions = descriptions[:len(column_names)] # truncate descriptions to match the number of columns
        df = df.iloc[2:].copy() # remove the first two rows
        df.columns = pd.MultiIndex.from_arrays([column_names, descriptions]) # set the first two rows as column names
        df = df.reset_index(drop=True) # reset the index
        #--- fixing accents
        
        return df
    except Exception as e:
        print(f"Error loading data from {file_path}: {e}")
        return None

# print("data_file: ", data_file)

# raw = pd.read_csv(data_file, sep="\t", encoding="latin-1", header=None) #change the delimiter to your needs.

# column_names = raw.iloc[0].tolist()
# descriptions = raw.iloc[1].tolist()

# descriptions = ["","",""] + descriptions
# descriptions = descriptions[:len(column_names)] # truncate descriptions to match the number of columns

# df = raw.iloc[2:].copy() # remove the first two rows
# df.columns = pd.MultiIndex.from_arrays([column_names, descriptions]) # set the first two rows as column names

df_linear = load_data(data_file)
df_non_linearity = load_data(non_linearity_file)


display(df_linear.head())
display(df_non_linearity.head())

Loading data from /content/international-project/FSIQ-from-ECG/data/HRV_Lineaire_Full.csv...
Loading data from /content/international-project/FSIQ-from-ECG/data/HRV_NonLineaire_Full.csv...


Numéro Monogramme         DDN Sexe Tabac Diabète HTA Maturation Pulm  \
                                 M/F   0/1     0/1 0/1             0/1   
0      1        GAL  11/09/2009    M     1       0   0               0   
1      2        JOG  22/09/2009    M     0       0   0               0   
2      2        JOG  22/09/2009    M     0       0   0               0   
3      2        JOG  22/09/2009    M     0       0   0               0   
4      2        JOG  22/09/2009    M     0       0   0               0   

  Prématuré     SFA  ... TIpsd (IncNight) VLFIpsd (IncNight)  \
        0/1 HIV 3-4  ...               ms               ms^2   
0         0       0  ...                -                  -   
1         0       0  ...                -                  -   
2         0       0  ...                -                  -   
3         0       0  ...                -                  -   
4         0       0  ...                -                  -   

  %VLFIpsd (IncNight) LFIpsd (IncNight) %LFIpsd (IncNight) HFIpsd (IncNight)  \
                 ms^2                 %               ms^2                 %   
0                   -                 -                  -                 -   
1                   -                 -                  -                 -   
2                   -                 -                  -                 -   
3                   -                 -                  -                 -   
4                   -                 -                  -                 -   

  %HFIpsd (IncNight) NaN        
                ms^2   %        
0                  -   -  -  -  
1                  -   -  -  -  
2                  -   -  -  -  
3                  -   -  -  -  
4                  -   -  -  -  

[5 rows x 118 columns]

Numéro Monogramme         DDN Sexe Tabac Diabète HTA Maturation Pulm  \
                                 M/F   0/1     0/1 0/1             0/1   
0      1        GAL  11/09/2009    M     1       0   0               0   
1      2        JOG  22/09/2009    M     0       0   0               0   
2      2        JOG  22/09/2009    M     0       0   0               0   
3      2        JOG  22/09/2009    M     0       0   0               0   
4      2        JOG  22/09/2009    M     0       0   0               0   

  Prématuré     SFA  ... CE (EntropyNight) CCE (EntropyNight)  \
        0/1 HIV 3-4  ...                 -                  -   
0         0       0  ...                 -                  -   
1         0       0  ...                 -                  -   
2         0       0  ...                 -                  -   
3         0       0  ...                 -                  -   
4         0       0  ...                 -                  -   

  NCCE (EntropyNight) rho (EntropyNight) LZC (Entropy24h) LZC (EntropyDay)  \
                    -                  -                -                -   
0                   -                  -                -                -   
1                   -                  -                -                -   
2                   -                  -                -                -   
3                   -                  -                -                -   
4                   -                  -                -                -   

  LZC (EntropyNight)      NaN        
                   -        -        
0                  -  0.99868  -  -  
1                  -   1.0303  -  -  
2                  -  0.92387  -  -  
3                  -  0.87586  -  -  
4                  -  0.77442  -  -  

[5 rows x 158 columns]

#### loading results

In [109]:
results_file = os.getenv("RESULTS_FILE")


def load_results(file_path):
    try:
        print(f"Loading results from {file_path}...")
        df_results = pd.read_csv(file_path, delimiter="\t", header=None) #change the delimiter to your needs.
        group_headers = df_results.iloc[0].fillna("")
        group_headers = (
            group_headers.replace("", pd.NA)
            .ffill()
            .fillna("")
        )
        column_names = df_results.iloc[1].tolist()
        real_df_results = df_results.iloc[2:].copy() # remove the first two rows
        real_df_results.columns = pd.MultiIndex.from_arrays([group_headers, column_names]) #
        real_df_results = real_df_results.reset_index(drop=True) # reset the index
        return real_df_results
    except Exception as e:
        print(f"Error loading results from {file_path}: {e}")
        return None 

print("results_file: ", results_file)

df_results = load_results(results_file)

display(df_results.head(10))

results_file:  /content/international-project/FSIQ-from-ECG/data/Résultats WPPSI III.csv
Loading results from /content/international-project/FSIQ-from-ECG/data/Résultats WPPSI III.csv...


0                           NS Verb.            NS Perf.      Note composite  \
   N°  NaN         DDN  Age      COM   inf  DIM      CUB  AOB            QIV   
0   1  GAL  11/09/2009  NaN      NaN   NaN  NaN      NaN  NaN            NaN   
1   2  JOG  22/09/2009   37       10  11.0    9        5   11            102   
2   3  DUL  26/09/2009  NaN      NaN   NaN  NaN      NaN  NaN            NaN   
3   4  DIM  28/09/2009   36        8   5.0    7        5    9             81   
4   5  ROM  30/09/2009   36       14  13.0    9        7    6            122   
5   6  PRG  03/10/2009   36       10  10.0   11       10   10             99   
6   7  DAA  05/10/2009   36       14  14.0   14        9   14            125   
7   8  DEJ  07/10/2009   36       11  16.0   11        7    8            122   
8   9  CHH  10/10/2009   38       10  10.0    9        7   10             99   
9  10  ZOY  27/07/2009  NaN      NaN   NaN  NaN      NaN  NaN            NaN   

0  ... Rang percentile      Intervalle de confiance (95%)                     \
   ...             QIT  NCL                           QIV    NaN  QIP    NaN   
0  ...             NaN  NaN                           NaN    NaN  NaN    NaN   
1  ...              37   42                            93  111.0   80  100.0   
2  ...             NaN  NaN                           NaN    NaN  NaN    NaN   
3  ...               7   16                            74   92.0   75   95.0   
4  ...              50   73                           111  128.0   73   93.0   
5  ...              50   58                            90  108.0   90  110.0   
6  ...              91   94                           113  131.0   97  118.0   
7  ...              61   66                           111  128.0   77   98.0   
8  ...              37   42                            90  108.0   83  103.0   
9  ...             NaN  NaN                           NaN    NaN  NaN    NaN   

0                          
   QIT    NaN  NCL    NaN  
0  NaN    NaN  NaN    NaN  
1   87  104.0   89  106.0  
2  NaN    NaN  NaN    NaN  
3   72   88.0   78   95.0  
4   92  108.0   99  117.0  
5   92  108.0   94  111.0  
6  110  126.0  112  129.0  
7   95  112.0   97  114.0  
8   87  104.0   89  106.0  
9  NaN    NaN  NaN    NaN  

[10 rows x 25 columns]

### Preprocessing

#### Target variable

We have to choose between those 4 variables:

- QIT (Quantitative Intelligence Test)
- QIV (Quantitative Intelligence Verbal)
- QIP (Quantitative Intelligence Performance)
- NCL (Nonverbal Cognitive Level)

For this analysis, we will use QIT as the target variable.

In [110]:
TARGET_VARIABLE = "QIT"

#### M0 Filterting

In [111]:
passage_positions = [
    i for i, column in enumerate(df_linear.columns)
    if column[0] == "Passage"
] 

print(df_linear.columns[passage_positions])#MultiIndex([('Passage', '-')],)

passage = (
    df_linear.iloc[:, passage_positions[0]]
    .astype("string")
    .str.strip()
) #this is the column that contains the passage information, we will use it to filter the data

df_linear_m0 = df_linear.loc[passage.eq("M0")].copy()

display(df_linear_m0.head())
print(df_linear_m0.shape)

df_non_linearity_m0 = df_non_linearity.loc[passage.eq("M0")].copy()
display(df_non_linearity_m0.head())
print(df_non_linearity_m0.shape)

MultiIndex([('Passage', '-')],
           )


Numéro Monogramme         DDN Sexe Tabac Diabète HTA Maturation Pulm  \
                                  M/F   0/1     0/1 0/1             0/1   
0       1        GAL  11/09/2009    M     1       0   0               0   
1       2        JOG  22/09/2009    M     0       0   0               0   
6       3        DUL  26/09/2009    M     0       0   0               0   
7       4        DIM  28/09/2009    M     0       1   1               0   
12      5        ROM  30/09/2009    M     0       0   0               0   

   Prématuré     SFA  ... TIpsd (IncNight) VLFIpsd (IncNight)  \
         0/1 HIV 3-4  ...               ms               ms^2   
0          0       0  ...                -                  -   
1          0       0  ...                -                  -   
6          0       0  ...                -                  -   
7          0       0  ...                -                  -   
12         0       0  ...                -                  -   

   %VLFIpsd (IncNight) LFIpsd (IncNight) %LFIpsd (IncNight) HFIpsd (IncNight)  \
                  ms^2                 %               ms^2                 %   
0                    -                 -                  -                 -   
1                    -                 -                  -                 -   
6                    -                 -                  -                 -   
7                    -                 -                  -                 -   
12                   -                 -                  -                 -   

   %HFIpsd (IncNight) NaN        
                 ms^2   %        
0                   -   -  -  -  
1                   -   -  -  -  
6                   -   -  -  -  
7                   -   -  -  -  
12                  -   -  -  -  

[5 rows x 118 columns]

(295, 118)


Numéro Monogramme         DDN Sexe Tabac Diabète HTA Maturation Pulm  \
                                  M/F   0/1     0/1 0/1             0/1   
0       1        GAL  11/09/2009    M     1       0   0               0   
1       2        JOG  22/09/2009    M     0       0   0               0   
6       3        DUL  26/09/2009    M     0       0   0               0   
7       4        DIM  28/09/2009    M     0       1   1               0   
12      5        ROM  30/09/2009    M     0       0   0               0   

   Prématuré     SFA  ... CE (EntropyNight) CCE (EntropyNight)  \
         0/1 HIV 3-4  ...                 -                  -   
0          0       0  ...                 -                  -   
1          0       0  ...                 -                  -   
6          0       0  ...                 -                  -   
7          0       0  ...                 -                  -   
12         0       0  ...                 -                  -   

   NCCE (EntropyNight) rho (EntropyNight) LZC (Entropy24h) LZC (EntropyDay)  \
                     -                  -                -                -   
0                    -                  -                -                -   
1                    -                  -                -                -   
6                    -                  -                -                -   
7                    -                  -                -                -   
12                   -                  -                -                -   

   LZC (EntropyNight)      NaN        
                    -        -        
0                   -  0.99868  -  -  
1                   -   1.0303  -  -  
6                   -   1.0188  -  -  
7                   -   1.0247  -  -  
12                  -   1.0242  -  -  

[5 rows x 158 columns]

(295, 158)


#### Remove rows of WSSPI with missing values in the target variable

In [112]:
df_results_clean = df_results.dropna(subset=[("Note composite", TARGET_VARIABLE)]).copy()

print(df_results_clean.shape)
print(df_results.shape)

(195, 25)
(302, 25)


#### Merge linear + nonlinear HRV on Monogramme + DDN + Passage

In [120]:
#convert multiindex to single index for easier merging
df_linear_m0_clean = df_linear_m0.copy()
df_linear_m0_clean.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_linear_m0_clean.columns]

display(df_linear_m0_clean.head())
#print every single column name in the dataframe, not truncated
print(df_linear_m0_clean.columns.to_list())

df_non_linearity_m0_clean = df_non_linearity_m0.copy()
df_non_linearity_m0_clean.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_non_linearity_m0_clean.columns]

#merge
df_merged = pd.merge(
    df_linear_m0_clean,
    df_non_linearity_m0_clean,
    on=["Monogramme", "DDN", "Passage_(-)"],
    how="inner",
)

display(df_merged.head())

,Numéro,Monogramme,DDN,Sexe_(M/F),Tabac_(0/1),Diabète_(0/1),HTA_(0/1),Maturation Pulm_(0/1),Prématuré_(0/1),SFA_(HIV 3-4),...,TIpsd (IncNight)_(ms),VLFIpsd (IncNight)_(ms^2),%VLFIpsd (IncNight)_(ms^2),LFIpsd (IncNight)_(%),%LFIpsd (IncNight)_(ms^2),HFIpsd (IncNight)_(%),%HFIpsd (IncNight)_(ms^2),nan_(%),nan,nan
0,1,GAL,11/09/2009,M,1,0,0,0,0,0,...,-,-,-,-,-,-,-,-,-,-
1,2,JOG,22/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,-,-,-
6,3,DUL,26/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,-,-,-
7,4,DIM,28/09/2009,M,0,1,1,0,0,0,...,-,-,-,-,-,-,-,-,-,-
12,5,ROM,30/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,-,-,-


['Numéro', 'Monogramme', 'DDN', 'Sexe_(M/F)', 'Tabac_(0/1)', 'Diabète_(0/1)', 'HTA_(0/1)', 'Maturation Pulm_(0/1)', 'Prématuré_(0/1)', 'SFA_(HIV 3-4)', 'R�a � la naissance_(LMPV)', 'Pathologie neurologique_(Dilatation ventr)', 'Sortie du service_(Exam neuro normal)', "D�pression: prise d'IRS_(Labilit� cardiaq)", 'Filename_(0/1)', 'Date_(Unit)', 'Recorder_(-)', 'Passage_(-)', 'Total duration_(Hz)', 'Day duration_(-)', 'Night duration', 'Night beginning_(-)', 'Night end_(-)', 'nb RR total_(-)', 'nb RR total day_(-)', 'nb RR total night_(-)', 'nb RR analyzed total_(-)', 'nb RR analyzed day_(-)', 'nb RR analyzed night_(-)', 'nb RR excluded total_(-)', 'nb RR excluded day_(-)', 'nb RR excluded night_(-)', 'nb RR corrected total_(-)', ' nb RR corrected day_(-)', 'nb RR corrected night_(-)', 'Mean RR (TD24h)_(-)', 'Mean HR (TD24h)_(ms)', 'NN20 (TD24h)_(bpm)', 'pcNN20 (TD24h)_(-)', 'NN30 (TD24h)_(%)', 'pcNN30 (TD24h)_(-)', 'NN50 (TD24h)_(%)', 'pcNN50 (TD24h)_(-)', 'SDNN (TD24h)_(%)', 'RMSSD (T

,Numéro_x,Monogramme,DDN,Sexe_(M/F)_x,Tabac_(0/1)_x,Diabète_(0/1)_x,HTA_(0/1)_x,Maturation Pulm_(0/1)_x,Prématuré_(0/1)_x,SFA_(HIV 3-4)_x,...,CE (EntropyNight)_(-),CCE (EntropyNight)_(-),NCCE (EntropyNight)_(-),rho (EntropyNight)_(-),LZC (Entropy24h)_(-),LZC (EntropyDay)_(-),LZC (EntropyNight)_(-),nan_(-),nan_y,nan_y
0,1,GAL,11/09/2009,M,1,0,0,0,0,0,...,-,-,-,-,-,-,-,0.99868,-,-
1,2,JOG,22/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,1.0303,-,-
2,3,DUL,26/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,1.0188,-,-
3,4,DIM,28/09/2009,M,0,1,1,0,0,0,...,-,-,-,-,-,-,-,1.0247,-,-
4,5,ROM,30/09/2009,M,0,0,0,0,0,0,...,-,-,-,-,-,-,-,1.0242,-,-


#### Merge HRV features with WSSPI data on Monogramme + DDN

In [127]:
#list of columns in df_results_clean
df_results_clean_tmp = df_results_clean.copy()
#df_results_clean_tmp.columns
print(df_results_clean_tmp.columns.to_list())

#for df_results, the monogramme is ('', 'N°'), the DDN is ('', 'DDN'), 
#convert multiindex to single index for easier merging
df_results_clean_tmp.columns = [str(col[0]) + "_(" + str(col[1]) + ")" if col[1] != "" else str(col[0]) for col in df_results_clean_tmp.columns]

print(df_results_clean_tmp.columns.to_list())

#edit column names in df_results_clean_tmp to match df_merged
df_results_clean_tmp = df_results_clean_tmp.rename(columns={
    "_(nan)": "Monogramme",
    "_(DDN)": "DDN",
})

#merge df_merged with df_results_clean_tmp on Monogramme and DDN
df_merged_full = pd.merge(
    df_merged,
    df_results_clean_tmp,
    on=["Monogramme", "DDN"],
    how="inner",
)

display(df_merged_full.head())
    

[('', 'N°'), ('', nan), ('', 'DDN'), ('', 'Age'), ('NS Verb.', 'COM'), ('NS Verb.', np.float64(inf)), ('NS Verb.', 'DIM'), ('NS Perf.', 'CUB'), ('NS Perf.', 'AOB'), ('Note composite', 'QIV'), ('Note composite', 'QIP'), ('Note composite', 'QIT'), ('Note composite', 'NCL'), ('Rang percentile', 'QIV'), ('Rang percentile', 'QIP'), ('Rang percentile', 'QIT'), ('Rang percentile', 'NCL'), ('Intervalle de confiance (95%)', 'QIV'), ('Intervalle de confiance (95%)', nan), ('Intervalle de confiance (95%)', 'QIP'), ('Intervalle de confiance (95%)', nan), ('Intervalle de confiance (95%)', 'QIT'), ('Intervalle de confiance (95%)', nan), ('Intervalle de confiance (95%)', 'NCL'), ('Intervalle de confiance (95%)', nan)]
['_(N°)', '_(nan)', '_(DDN)', '_(Age)', 'NS Verb._(COM)', 'NS Verb._(inf)', 'NS Verb._(DIM)', 'NS Perf._(CUB)', 'NS Perf._(AOB)', 'Note composite_(QIV)', 'Note composite_(QIP)', 'Note composite_(QIT)', 'Note composite_(NCL)', 'Rang percentile_(QIV)', 'Rang percentile_(QIP)', 'Rang perce

,Numéro_x,Monogramme,DDN,Sexe_(M/F)_x,Tabac_(0/1)_x,Diabète_(0/1)_x,HTA_(0/1)_x,Maturation Pulm_(0/1)_x,Prématuré_(0/1)_x,SFA_(HIV 3-4)_x,...,Rang percentile_(QIT),Rang percentile_(NCL),Intervalle de confiance (95%)_(QIV),Intervalle de confiance (95%)_(nan),Intervalle de confiance (95%)_(QIP),Intervalle de confiance (95%)_(nan),Intervalle de confiance (95%)_(QIT),Intervalle de confiance (95%)_(nan),Intervalle de confiance (95%)_(NCL),Intervalle de confiance (95%)_(nan)
0,2,JOG,22/09/2009,M,0,0,0,0,0,0,...,37,42,93,111.0,80,100.0,87,104.0,89,106.0
1,4,DIM,28/09/2009,M,0,1,1,0,0,0,...,7,16,74,92.0,75,95.0,72,88.0,78,95.0
2,5,ROM,30/09/2009,M,0,0,0,0,0,0,...,50,73,111,128.0,73,93.0,92,108.0,99,117.0
3,12,CLT,16/10/2009,M,1,0,0,0,0,0,...,25,58,85,103.0,80,100.0,83,99.0,94,111.0
4,14,GUS,21/10/2009,M,0,0,0,0,0,0,...,25,73,93,111.0,73,93.0,83,99.0,99,117.0
